# Frame Optimization Analysis

This notebook loads data for a specific frame number and recreates the global optimization scenario for that frame. It includes:
- Loading frame data by frame number
- Recreating the exact optimization scenario from the pipeline
- Per-iteration optimization tracking and visualization
- Comprehensive analysis and visualizations similar to the debug notebook


In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.spatial.transform import Rotation as R
from easydict import EasyDict as edict
import gtsam
from typing import List, Dict, Tuple, Optional

# Add project root to path
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.append(project_root)

# Import pipeline components
from point2pose.pipeline.components.local_optimizer import LocalOptimizer
from point2pose.pipeline.components.key_frame_graph import KeyFrameGraph
from point2pose.data_types.object_frame_data import ObjectFrameData
from point2pose.data_types.key_frame import KeyFrame
from point2pose.modules.object.object import Object
from point2pose.data_types.point_track_table import PointTrackTable
from point2pose.utils.transform import inverse_SE3, transform_pts

# Use interactive backend for rotatable 3D plots
try:
    from IPython import get_ipython
    ipython = get_ipython()
    if ipython is not None:
        ipython.run_line_magic('matplotlib', 'widget')
except:
    try:
        if ipython is not None:
            ipython.run_line_magic('matplotlib', 'notebook')
    except:
        import matplotlib
        matplotlib.use('inline')
        print("Note: Interactive 3D plots not available. Using static plots.")


In [ ]:
def find_meta_data_path(
    register_folder: str = None,
    results_dir: Optional[str] = None,
    video_name: Optional[str] = None,
) -> Optional[str]:
    """Find meta_data.npz file in expected locations."""
    script_dir = os.path.dirname(os.path.abspath('.'))
    project_root = os.path.abspath('..')
    
    meta_data_paths = []
    
    # Check new results folder structure first
    if results_dir and video_name:
        new_structure_path = os.path.join(results_dir, video_name, "meta_data", "meta_data.npz")
        meta_data_paths.append(new_structure_path)
        default_results_dir = os.path.join(project_root, "results", "ho3d_single")
        if results_dir != default_results_dir:
            meta_data_paths.append(os.path.join(default_results_dir, video_name, "meta_data", "meta_data.npz"))
    
    # Existing paths for backward compatibility
    if register_folder:
        meta_data_paths.extend([
            os.path.join(register_folder, "meta_data.npz"),
            os.path.join(os.path.dirname(register_folder), "meta_data.npz"),
        ])
    
    for path in meta_data_paths:
        if os.path.exists(path):
            return path
    
    return None

def load_metadata(path):
    """Load metadata from NPZ file."""
    if not os.path.exists(path):
        raise FileNotFoundError(f"{path} does not exist")
    data = np.load(path, allow_pickle=True)
    return data

def get_slice(data, key_prefix, idx, reshape_dim=None):
    """Helper to extract a slice from ragged array storage."""
    try:
        offsets = data[f"{key_prefix}_offsets"]
        lengths = data[f"{key_prefix}_lengths"]
        start = offsets[idx]
        length = lengths[idx]
        arr = data[f"{key_prefix}_data"][start:start+length]
        if reshape_dim:
            if arr.size == 0:
                return arr.reshape(0, reshape_dim)
            return arr.reshape(-1, reshape_dim)
        return arr
    except KeyError:
        return np.array([])

def get_frame_data(data, idx):
    """Extracts ObjectFrameData components for a specific frame index."""
    # 3D points observed in camera frame (from registration)
    cur_3d = get_slice(data, 'reg_curr3d', idx, 3)
    
    # Indices of these points (Track IDs)
    cur_3d_idx = get_slice(data, 'reg_key_points_idx', idx)
    
    # Valid indices used by the registration
    valid_idx = get_slice(data, 'reg_valid_idx', idx)
    if valid_idx.size == 0:
        valid_idx = cur_3d_idx.copy()
    
    # Inliers/Residuals from registration
    inliers = get_slice(data, 'reg_inliers', idx)
    if inliers.size > 0:
        inliers = inliers.astype(bool)
    else:
        inliers = np.ones(cur_3d.shape[0], dtype=bool) if cur_3d.shape[0] > 0 else np.array([], dtype=bool)
        
    residuals = get_slice(data, 'reg_residuals', idx)
    if residuals.size == 0:
        residuals = np.zeros(cur_3d.shape[0]) if cur_3d.shape[0] > 0 else np.array([])
    
    # Uncertainties
    uncertainties = get_slice(data, 'uncertainties', idx)
    if uncertainties.size == 0:
        uncertainties = np.ones(cur_3d.shape[0]) * 0.1 if cur_3d.shape[0] > 0 else np.array([])
    
    # Poses at different stages
    if 'pose_frontend' in data:
        pose_frontend = data['pose_frontend'][idx]
    else:
        print(f"Warning: pose_frontend not found in metadata for frame {idx}, using obj_pose")
        pose_frontend = data['obj_pose'][idx] if 'obj_pose' in data else data['obj_init_pose'][idx]
    
    if 'pose_local' in data:
        pose_local = data['pose_local'][idx]
    else:
        pose_local = pose_frontend
    
    # GT Pose (if available)
    gt_pose = data['obj_pose'][idx] if 'obj_pose' in data else None
    
    # Frame ID
    frame_id = int(data['frame_id'][idx])
    
    # Is keyframe?
    is_keyframe = data['is_key_frame'][idx] if 'is_key_frame' in data else False
    
    return {
        'frame_id': frame_id,
        'pose_frontend': pose_frontend,
        'pose_local': pose_local,
        'gt_pose': gt_pose,
        'cur_3d': cur_3d,
        'cur_3d_idx': cur_3d_idx,
        'valid_idx': valid_idx,
        'inliers': inliers,
        'residuals': residuals,
        'uncertainties': uncertainties,
        'is_keyframe': is_keyframe
    }


In [ ]:
class IterationTrackingOptimizer:
    """Wrapper around LMGraphOptimizer that tracks per-iteration data."""
    
    def __init__(self, base_optimizer):
        self.base_optimizer = base_optimizer
        self.iteration_data = []
        # Don't copy graph/values here - they may not be initialized yet
        # We'll copy them in optimize_with_tracking after ensuring base_optimizer has the correct state
        self.graph = None
        self.values = None
        
    def optimize_with_tracking(self, data: ObjectFrameData):
        """Run optimization and track per-iteration changes."""
        self.iteration_data = []
        
        # Ensure base_optimizer has graph and values (they should already exist)
        if not hasattr(self.base_optimizer, '_graph'):
            self.base_optimizer._graph = gtsam.NonlinearFactorGraph()
        if not hasattr(self.base_optimizer, '_values'):
            self.base_optimizer._values = gtsam.Values()
        
        # NOW copy the graph and values for tracking (they should be populated by now)
        graph = gtsam.NonlinearFactorGraph(self.base_optimizer._graph)
        values = gtsam.Values(self.base_optimizer._values)
        
        # Store for potential later use
        self.graph = graph
        self.values = values
        
        frame_id = data.frame_id
        Xi = gtsam.symbol("x", frame_id)
        
        # Check if frame is already in graph
        frame_already_added = Xi in self.base_optimizer._inserted_poses
        
        # If frame is already added, update the pose value for re-optimization
        if frame_already_added:
            cur_pose_c2w = inverse_SE3(data.pose)
            cur_pose_c2w_gtsam = gtsam.Pose3(cur_pose_c2w)
            if values.exists(Xi):
                values.update(Xi, cur_pose_c2w_gtsam)
            else:
                values.insert(Xi, cur_pose_c2w_gtsam)
        
        if not frame_already_added:
            # Add the frame to the graph (replicate LMGraphOptimizer logic)
            cur_pose_c2w = inverse_SE3(data.pose)
            cur_pose_c2w_gtsam = gtsam.Pose3(cur_pose_c2w)
            
            # Insert pose variable
            values.insert(Xi, cur_pose_c2w_gtsam)
            self.base_optimizer._inserted_poses.add(Xi)
            
            # Add prior on first pose or between factor
            if not self.base_optimizer._initialized:
                graph.push_back(
                    gtsam.PriorFactorPose3(Xi, cur_pose_c2w_gtsam, self.base_optimizer._prior_noise)
                )
            elif data.rel_pose is not None:
                prev_id = self.base_optimizer._prev_frame_id
                X_prev = gtsam.symbol("x", prev_id)
                rel_T_cim12ci = gtsam.Pose3(inverse_SE3(data.rel_pose))
                # Note: between factors are commented out in the original, so we skip them
            
            # Insert landmark variables and factors
            if data.inliers.size > 0:
                seed_pose = cur_pose_c2w_gtsam
                if values.exists(Xi):
                    try:
                        seed_pose = values.atPose3(Xi)
                    except RuntimeError:
                        seed_pose = cur_pose_c2w_gtsam
                
                for m, lid in enumerate(data.valid_idx):
                    if not data.inliers[m] or np.isnan(data.cur_3d[m]).any():
                        continue
                    if data.residuals[m] > 0.001:
                        continue
                    
                    z_cam = data.cur_3d[m]
                    Lj = gtsam.symbol("l", int(lid))
                    
                    sigma_point = float(max(1e-4, data.residuals[m]))
                    base_noise = gtsam.noiseModel.Diagonal.Sigmas(
                        np.array([sigma_point * 10, sigma_point * 10, sigma_point * 10], dtype=float)
                    )
                    point_noise = gtsam.noiseModel.Robust(
                        gtsam.noiseModel.mEstimator.Huber(1.345), base_noise
                    )
                    
                    # Create landmark if missing
                    if Lj not in self.base_optimizer._inserted_landmarks:
                        pw = seed_pose.transformFrom(gtsam.Point3(*z_cam))
                        values.insert(Lj, pw)
                        self.base_optimizer._inserted_landmarks.add(Lj)
                        import bisect
                        bisect.insort(self.base_optimizer.inserted_landmark_ids, int(lid))
                    
                    z_range = float(np.linalg.norm(z_cam))
                    if z_range <= 1e-9:
                        continue
                    
                    z_bearing = gtsam.Unit3(z_cam / z_range)
                    graph.push_back(
                        gtsam.BearingRangeFactor3D(Xi, Lj, z_bearing, z_range, point_noise)
                    )
            
            # Update bookkeeping
            self.base_optimizer._prev_pose_inv = data.pose
            self.base_optimizer._prev_frame_id = frame_id
        
        # Handle first frame (not initialized yet)
        # The first frame gets added with a prior but doesn't get optimized
        if not self.base_optimizer._initialized:
            # For the first frame, we add it but don't optimize yet
            # Record the initial state
            initial_error = graph.error(values) if graph.size() > 0 else 0.0
            initial_pose = None
            if values.exists(Xi):
                initial_pose = values.atPose3(Xi).matrix()
            self.iteration_data.append({
                'iteration': 0,
                'error': initial_error,
                'values': initial_pose,
                'note': 'First frame - no optimization performed, only prior added'
            })
            # Update the base optimizer's state
            self.base_optimizer._graph = graph
            self.base_optimizer._values = values
            self.base_optimizer._initialized = True
            # Return None to indicate no optimization was performed
            # But we still have iteration_data with the initial state
            return None
        
        # Now track iterations
        # Get initial error and pose
        # Check if graph has factors before computing error
        if graph.size() == 0:
            print(f"Warning: Graph is empty after adding frame {frame_id}, cannot optimize")
            return None
            
        initial_error = graph.error(values)
        initial_pose = None
        if values.exists(Xi):
            initial_pose = values.atPose3(Xi).matrix()
        
        self.iteration_data.append({
            'iteration': 0,
            'error': initial_error,
            'values': initial_pose
        })
        
        # Create optimizer for iteration tracking
        lm_params = self.base_optimizer._lm_params
        max_iterations = lm_params.getMaxIterations()
        
        # Check if graph has any factors before optimizing
        if graph.size() == 0:
            print(f"Warning: Graph is empty, cannot optimize frame {frame_id}")
            return None
        
        try:
            optimizer = gtsam.LevenbergMarquardtOptimizer(graph, values, lm_params)
        except RuntimeError as e:
            print(f"Warning: Failed to create optimizer: {e}")
            return None
        
        # Track iterations by manually calling iterate()
        current_values = values
        for i in range(max_iterations):
            try:
                current_error = graph.error(current_values)
                
                # Perform one iteration
                optimizer.iterate()
                new_values = optimizer.values()
                new_error = graph.error(new_values)
                
                # Extract pose if available
                pose_matrix = None
                if new_values.exists(Xi):
                    pose_matrix = new_values.atPose3(Xi).matrix()
                
                error_change = current_error - new_error
                self.iteration_data.append({
                    'iteration': i + 1,
                    'error': new_error,
                    'error_change': error_change,
                    'values': pose_matrix
                })
                
                current_values = new_values
                
                # Check convergence
                if current_error > 0 and abs(error_change) < lm_params.getRelativeErrorTol() * current_error:
                    break
                if abs(error_change) < lm_params.getAbsoluteErrorTol():
                    break
                    
            except RuntimeError as e:
                # Optimization converged or failed
                print(f"Warning: Optimization iteration {i+1} failed: {e}")
                break
        
        # Update base optimizer's graph and values with the optimized results
        # Only update if we actually ran optimization (have iteration data)
        if len(self.iteration_data) > 1:  # More than just initial state
            self.base_optimizer._graph = graph  # Update graph (may have new factors)
            self.base_optimizer._values = current_values  # Update values with optimized results
        
        # Extract optimized pose
        if current_values.exists(Xi):
            Xi_hat = current_values.atPose3(Xi)
            pose_opt = inverse_SE3(Xi_hat.matrix())
            
            # Extract landmarks
            num_L = len(self.base_optimizer.inserted_landmark_ids)
            landmark_xyz = np.empty((num_L, 3), dtype=float)
            ids = np.empty((num_L,), dtype=np.int64)
            
            k = 0
            for lid in self.base_optimizer.inserted_landmark_ids:
                Lj = gtsam.symbol("l", int(lid))
                if current_values.exists(Lj):
                    p = np.asarray(current_values.atPoint3(Lj), dtype=float).reshape(3,)
                    landmark_xyz[k, :] = p
                    ids[k] = int(lid)
                    k += 1
            
            landmark_xyz = landmark_xyz[:k, :]
            ids = ids[:k]
            
            from point2pose.data_types.optimizer_result import OptimizerResult
            return OptimizerResult(
                obj_id=data.obj_id,
                frame_id=data.frame_id,
                pose_optimized=pose_opt,
                key_points_optimized=landmark_xyz,
                key_points_idx_optimized=ids,
            )
        
        return None


## Configuration

<!-- Set the frame number and metadata path here. -->


In [ ]:
# --- CONFIGURATION ---
FRAME_NUMBER = 89  # Change this to analyze a different frame

# Path to metadata file (adjust as needed)
results_dir = '/home/justin/code/point-to-pose/results/ho3d_single'
video_name = 'MPM10'
meta_data_path = os.path.join(results_dir, video_name, 'meta_data', 'meta_data.npz')

# Fallback paths
if not os.path.exists(meta_data_path):
    print(f"Path {meta_data_path} not found. Trying alternatives...")
    meta_data_path = find_meta_data_path(results_dir=results_dir, video_name=video_name)
    if meta_data_path is None:
        # Try default location
        meta_data_path = '/home/justin/code/point-to-pose/results/ho3d_single/MPM10/meta_data/meta_data.npz'

if not os.path.exists(meta_data_path):
    raise FileNotFoundError(f"Could not find metadata file. Please set meta_data_path manually.")

print(f"Loading metadata from: {meta_data_path}")
meta_data = load_metadata(meta_data_path)
num_frames = len(meta_data['frame_id'])
print(f"Loaded {num_frames} frames.")

# Check if frame exists
frame_ids = meta_data['frame_id']
if FRAME_NUMBER not in frame_ids:
    print(f"Frame {FRAME_NUMBER} not found in metadata.")
    print(f"Available frames: {frame_ids[:20]}..." if len(frame_ids) > 20 else f"Available frames: {frame_ids}")
    raise ValueError(f"Frame {FRAME_NUMBER} not found")

# Find frame index
frame_idx = None
for i, fid in enumerate(frame_ids):
    if fid == FRAME_NUMBER:
        frame_idx = i
        break

print(f"Frame {FRAME_NUMBER} found at index {frame_idx}")

# Check for keyframes in the metadata to help user pick a good frame
if 'is_key_frame' in meta_data:
    keyframe_indices = []
    for i in range(min(frame_idx + 50, len(meta_data['frame_id']))):  # Check up to frame_idx + 50 or end
        if meta_data['is_key_frame'][i]:
            keyframe_indices.append(i)
    if len(keyframe_indices) > 0:
        keyframe_frame_ids_list = [meta_data['frame_id'][i] for i in keyframe_indices]
        print(f"\nFound {len(keyframe_indices)} keyframes in metadata (up to frame {frame_idx + 50}):")
        print(f"  Keyframe indices: {keyframe_indices[:10]}{'...' if len(keyframe_indices) > 10 else ''}")
        print(f"  Keyframe frame IDs: {keyframe_frame_ids_list[:10]}{'...' if len(keyframe_frame_ids_list) > 10 else ''}")
        if FRAME_NUMBER in keyframe_frame_ids_list:
            kf_pos = keyframe_frame_ids_list.index(FRAME_NUMBER)
            if kf_pos == 0:
                print(f"\n⚠️  Frame {FRAME_NUMBER} is the FIRST keyframe - it will only get a prior factor, no optimization iterations.")
                print(f"   To see optimization iterations, try a later keyframe like: {keyframe_frame_ids_list[1] if len(keyframe_frame_ids_list) > 1 else 'a later frame'}")

# Create config structure matching pipeline
pipeline_cfg = edict({
    'local_optimizer': edict({
        'type': 'lm_graph',
        'params': edict({
            'local_graph_max_num_frames': -1,
            'max_iterations': 20,
            'relative_error_tol': 1e-5,
            'absolute_error_tol': 1e-5,
            'prior_noise_param': [0.1, 0.1, 0.1, 0.1, 0.1, 0.1]
        })
    }),
    'global_optimizer': edict({
        'type': 'lm_graph',
        'params': edict({
            'relinearize_threshold': 0.1,
            'relinearize_skip': 1,
            'max_iterations': 20,
            'relative_error_tol': 1e-5,
            'absolute_error_tol': 1e-5,
            'prior_noise_param': [0.01, 0.01, 0.01, 0.01, 0.01, 0.01]
        })
    })
})


## Recreate Optimization Scenario

This section recreates the optimization scenario up to the target frame, building the graph incrementally as the pipeline does.


In [ ]:
# Initialize pipeline components
local_optimizer = LocalOptimizer(pipeline_cfg)
kf_graph = KeyFrameGraph(pipeline_cfg)

# Create mock objects for state tracking
mock_obj = Object(0)
mock_obj.pose = np.eye(4)
mock_track_table = PointTrackTable.new(n0=0)
mock_track_table.obj2track_map = {0: []}

# Storage for trajectories and keyframes
traj_frontend = []
traj_local = []
traj_global = []
traj_gt = []
keyframes_list = []
kf_idx_counter = 0
keyframe_frame_ids = []

# Recreate pipeline up to (but not including) target frame
# We'll track iterations when we add the target frame
print(f"Recreating pipeline optimization up to (but not including) frame {FRAME_NUMBER}...")

for i in range(frame_idx):  # Stop before target frame
    fd = get_frame_data(meta_data, i)
    
    # Compute relative pose
    if i == 0:
        rel_pose = np.eye(4)
    else:
        prev_fd = get_frame_data(meta_data, i-1)
        prev_pose = prev_fd['pose_frontend']
        cur_pose = fd['pose_frontend']
        prev_pose_inv = inverse_SE3(prev_pose)
        rel_pose = cur_pose @ prev_pose_inv
    
    # Store frontend pose
    traj_frontend.append(fd['pose_frontend'].copy())
    
    # Local optimization
    object_frame_data = ObjectFrameData(
        obj_id=0,
        frame_id=fd['frame_id'],
        pose=fd['pose_frontend'],
        rel_pose=rel_pose,
        cur_3d=fd['cur_3d'],
        cur_3d_idx=fd['cur_3d_idx'],
        inliers=fd['inliers'],
        residuals=fd['residuals'],
        valid_idx=fd['valid_idx'],
        uncertainties=fd['uncertainties']
    )
    
    opt_result = local_optimizer.optimize(object_frame_data)
    
    if opt_result is not None:
        mock_obj.pose = opt_result.pose_optimized.copy()
        if len(fd['cur_3d_idx']) > 0:
            mock_track_table.obj2track_map[0] = fd['cur_3d_idx'].tolist()
        traj_local.append(mock_obj.pose.copy())
    else:
        mock_obj.pose = fd['pose_frontend'].copy()
        traj_local.append(fd['pose_frontend'].copy())
    
    # Keyframe handling
    if fd['is_keyframe']:
        keyframe_frame_ids.append(fd['frame_id'])
        
        num_obs = len(fd['cur_3d'])
        if len(fd['uncertainties']) > num_obs and len(fd['cur_3d_idx']) > 0:
            try:
                obs_uncertainties = fd['uncertainties'][fd['cur_3d_idx']]
            except:
                obs_uncertainties = np.ones(num_obs, dtype=float) * 0.1
        else:
            obs_uncertainties = np.ones(num_obs, dtype=float) * 0.1
        
        kf = KeyFrame(
            frame_id=fd['frame_id'],
            obj_id=0,
            kf_idx=kf_idx_counter,
            timestamp=None,
            pose=mock_obj.pose.copy(),
            kp_track_indices=fd['cur_3d_idx'] if num_obs > 0 else np.array([], dtype=int),
            kp_2d=np.zeros((num_obs, 2)) if num_obs > 0 else np.zeros((0, 2)),
            kp_3d_camera=fd['cur_3d'] if num_obs > 0 else np.zeros((0, 3)),
            kp_3d_object=np.zeros((num_obs, 3)) if num_obs > 0 else np.zeros((0, 3)),
            kp_valid=np.ones(num_obs, dtype=bool) if num_obs > 0 else np.array([], dtype=bool),
            obs_track_indices=fd['cur_3d_idx'] if num_obs > 0 else np.array([], dtype=int),
            obs_2d=np.zeros((num_obs, 2)) if num_obs > 0 else np.zeros((0, 2)),
            obs_3d_camera=fd['cur_3d'] if num_obs > 0 else np.zeros((0, 3)),
            obs_3d_object=np.zeros((num_obs, 3)) if num_obs > 0 else np.zeros((0, 3)),
            obs_valid=np.ones(num_obs, dtype=bool) if num_obs > 0 else np.array([], dtype=bool),
            obs_visible=np.ones(num_obs, dtype=bool) if num_obs > 0 else np.array([], dtype=bool),
            obs_uncertainties=obs_uncertainties,
            reg_inliers=fd['inliers'],
            reg_residuals=fd['residuals'],
            reg_valid_idx=fd['valid_idx'],
            dense_pts=np.zeros((0, 3))
        )
        keyframes_list.append(kf)
        kf_idx_counter += 1
        
        # Reset local optimizer on keyframe
        local_optimizer.reset(0)
    
    # Global optimization (only for keyframes)
    if keyframes_list and fd['is_keyframe']:
        # Get optimizer before update to verify it's the same instance
        opt_before = kf_graph._get_optimizer(0)
        opt_before_id = id(opt_before)
        opt_before_graph_size = opt_before._graph.size() if hasattr(opt_before, '_graph') else 0
        
        updated_poses, updated_landmarks = kf_graph.update(keyframes_list)
        
        # Verify optimizer after update
        opt_after = kf_graph._get_optimizer(0)
        opt_after_id = id(opt_after)
        opt_after_graph_size = opt_after._graph.size() if hasattr(opt_after, '_graph') else 0
        
        # Debug: Check if optimizer instance persisted
        if opt_before_id == opt_after_id:
            print(f"  [Frame {fd['frame_id']}] Optimizer instance persisted (id={opt_before_id}), graph size: {opt_before_graph_size} -> {opt_after_graph_size}")
        else:
            print(f"  [Frame {fd['frame_id']}] WARNING: Optimizer instance changed! Before: {opt_before_id}, After: {opt_after_id}")
        
        for kf in keyframes_list:
            key = (kf.obj_id, kf.kf_idx)
            if key in updated_poses:
                mock_obj.pose = updated_poses[key].copy()
        keyframes_list = []
    
    traj_global.append(mock_obj.pose.copy())
    
    # Store GT
    if fd['gt_pose'] is not None:
        traj_gt.append(fd['gt_pose'].copy())
    else:
        if len(traj_gt) == 0:
            traj_gt.append(np.eye(4))
        else:
            traj_gt.append(traj_gt[-1].copy())

print(f"Completed optimization loop. Processed {frame_idx} frames (up to but not including frame {FRAME_NUMBER}).")
print(f"Found {kf_idx_counter} keyframes before frame {FRAME_NUMBER}.")
if len(keyframe_frame_ids) > 0:
    print(f"Keyframe Frame IDs before target: {keyframe_frame_ids}")


## Per-Iteration Optimization Analysis

Now we'll recreate the optimization for the target frame with per-iteration tracking.


In [ ]:
# Get frame data for target frame
target_fd = get_frame_data(meta_data, frame_idx)

# Check if this is a keyframe
is_target_keyframe = target_fd['is_keyframe']
print(f"\nFrame {FRAME_NUMBER} is {'a KEYFRAME' if is_target_keyframe else 'NOT a keyframe'}")

# Now we'll add the target frame and track iterations
# The graph is built up to (but not including) the target frame

# Select the appropriate optimizer
if is_target_keyframe:
    # For keyframes, we need to add to global optimizer
    # Get the optimizer (it should already exist if keyframes were processed)
    tracking_opt = kf_graph._get_optimizer(0)
    
    # Debug: Check the optimizer state BEFORE copying
    print(f"\n=== Optimizer State Check ===")
    print(f"tracking_opt type: {type(tracking_opt)}")
    print(f"tracking_opt id: {id(tracking_opt)}")
    print(f"Has _graph attribute: {hasattr(tracking_opt, '_graph')}")
    print(f"Has _values attribute: {hasattr(tracking_opt, '_values')}")
    
    if hasattr(tracking_opt, '_graph'):
        print(f"Graph size: {tracking_opt._graph.size()}")
        print(f"Graph type: {type(tracking_opt._graph)}")
    else:
        print("ERROR: tracking_opt does not have _graph attribute!")
        
    if hasattr(tracking_opt, '_values'):
        print(f"Values size: {tracking_opt._values.size()}")
        print(f"Values type: {type(tracking_opt._values)}")
    else:
        print("ERROR: tracking_opt does not have _values attribute!")
    
    if hasattr(tracking_opt, '_inserted_poses'):
        print(f"Inserted poses: {len(tracking_opt._inserted_poses)}")
        if len(tracking_opt._inserted_poses) > 0:
            print(f"  Pose symbols: {sorted([int(gtsam.Symbol(s).index()) for s in tracking_opt._inserted_poses])}")
    else:
        print("ERROR: tracking_opt does not have _inserted_poses attribute!")
    
    if hasattr(tracking_opt, '_initialized'):
        print(f"Initialized: {tracking_opt._initialized}")
    else:
        print("ERROR: tracking_opt does not have _initialized attribute!")
    
    # Check if optimizer was actually used (has graph/values)
    # If not, it means no keyframes were processed before this one
    if not hasattr(tracking_opt, '_graph') or tracking_opt._graph.size() == 0:
        print(f"\n⚠️  Warning: Global optimizer is empty. Frame {FRAME_NUMBER} appears to be the first keyframe.")
        print("The optimizer will be initialized when we add this frame.")
    else:
        print(f"\n✓ Optimizer has graph with {tracking_opt._graph.size()} factors and {tracking_opt._values.size()} values")
    
    # Create KeyFrame for target frame
    num_obs = len(target_fd['cur_3d'])
    if len(target_fd['uncertainties']) > num_obs and len(target_fd['cur_3d_idx']) > 0:
        try:
            obs_uncertainties = target_fd['uncertainties'][target_fd['cur_3d_idx']]
        except:
            obs_uncertainties = np.ones(num_obs, dtype=float) * 0.1
    else:
        obs_uncertainties = np.ones(num_obs, dtype=float) * 0.1
    
    # Use the pose from local optimization (or frontend if not optimized yet)
    target_kf = KeyFrame(
        frame_id=target_fd['frame_id'],
        obj_id=0,
        kf_idx=kf_idx_counter,
        timestamp=None,
        pose=mock_obj.pose.copy(),  # Current pose from previous frames
        kp_track_indices=target_fd['cur_3d_idx'] if num_obs > 0 else np.array([], dtype=int),
        kp_2d=np.zeros((num_obs, 2)) if num_obs > 0 else np.zeros((0, 2)),
        kp_3d_camera=target_fd['cur_3d'] if num_obs > 0 else np.zeros((0, 3)),
        kp_3d_object=np.zeros((num_obs, 3)) if num_obs > 0 else np.zeros((0, 3)),
        kp_valid=np.ones(num_obs, dtype=bool) if num_obs > 0 else np.array([], dtype=bool),
        obs_track_indices=target_fd['cur_3d_idx'] if num_obs > 0 else np.array([], dtype=int),
        obs_2d=np.zeros((num_obs, 2)) if num_obs > 0 else np.zeros((0, 2)),
        obs_3d_camera=target_fd['cur_3d'] if num_obs > 0 else np.zeros((0, 3)),
        obs_3d_object=np.zeros((num_obs, 3)) if num_obs > 0 else np.zeros((0, 3)),
        obs_valid=np.ones(num_obs, dtype=bool) if num_obs > 0 else np.array([], dtype=bool),
        obs_visible=np.ones(num_obs, dtype=bool) if num_obs > 0 else np.array([], dtype=bool),
        obs_uncertainties=obs_uncertainties,
        reg_inliers=target_fd['inliers'],
        reg_residuals=target_fd['residuals'],
        reg_valid_idx=target_fd['valid_idx'],
        dense_pts=np.zeros((0, 3))
    )
    
    # For keyframes, we need to manually track the optimization
    # The kf_graph.update() method calls the optimizer internally
    # We'll need to intercept that call or manually call the optimizer
    
    # Recreate ObjectFrameData as kf_graph.update() would create it
    obj_id = target_kf.obj_id
    kf_idx = target_kf.kf_idx
    cur_pose = np.asarray(target_kf.pose, dtype=float)
    
    # Compute relative pose
    if obj_id in kf_graph._last_kf_pose:
        prev_pose = kf_graph._last_kf_pose[obj_id]
        prev_pose_inv = inverse_SE3(prev_pose)
        rel_pose = cur_pose @ prev_pose_inv
    else:
        rel_pose = np.eye(4, dtype=float)
    
    # Build measurement data
    if (target_kf.obs_3d_camera is not None and target_kf.obs_3d_camera.size > 0 and
        target_kf.obs_valid is not None and target_kf.obs_visible is not None):
        valid_mask = np.asarray(target_kf.obs_valid, dtype=bool) & np.asarray(target_kf.obs_visible, dtype=bool)
        if np.any(valid_mask):
            cur_3d = np.asarray(target_kf.obs_3d_camera[valid_mask], dtype=float)
            cur_3d_idx = np.asarray(target_kf.obs_track_indices[valid_mask], dtype=int)
            if target_kf.obs_uncertainties is not None:
                uncertainties = np.asarray(target_kf.obs_uncertainties[valid_mask], dtype=float)
            else:
                uncertainties = 0.5 * np.ones((cur_3d.shape[0],), dtype=float)
        else:
            cur_3d = np.zeros((0, 3), dtype=float)
            cur_3d_idx = np.zeros((0,), dtype=int)
            uncertainties = np.zeros((0,), dtype=float)
    else:
        cur_3d = np.zeros((0, 3), dtype=float)
        cur_3d_idx = np.zeros((0,), dtype=int)
        uncertainties = np.zeros((0,), dtype=float)
    
    target_object_frame_data = ObjectFrameData(
        obj_id=obj_id,
        frame_id=kf_idx,
        pose=cur_pose,
        rel_pose=rel_pose,
        cur_3d=cur_3d,
        cur_3d_idx=cur_3d_idx,
        valid_idx=target_kf.reg_valid_idx,
        inliers=target_kf.reg_inliers,
        residuals=target_kf.reg_residuals,
        uncertainties=uncertainties,
    )

iteration_tracker = IterationTrackingOptimizer(tracking_opt)


# ALWAYS copy the state from tracking_opt to iteration_tracker.base_optimizer
# This ensures we're working with the same state, even if empty
print(f"\nCopying optimizer state from kf_graph optimizer...")

# Since base_optimizer IS tracking_opt, they share the same state
# Just verify the state exists and show it
if hasattr(tracking_opt, '_graph'):
    print(f"  Graph size: {tracking_opt._graph.size()}")
    print(f"  Graph type: {type(tracking_opt._graph)}")
else:
    print("  ERROR: tracking_opt does not have _graph attribute!")
    tracking_opt._graph = gtsam.NonlinearFactorGraph()

if hasattr(tracking_opt, '_values'):
    print(f"  Values size: {tracking_opt._values.size()}")
    print(f"  Values type: {type(tracking_opt._values)}")
else:
    print("  ERROR: tracking_opt does not have _values attribute!")
    tracking_opt._values = gtsam.Values()

if hasattr(tracking_opt, '_inserted_poses'):
    print(f"  Inserted poses: {len(tracking_opt._inserted_poses)}")
    if len(tracking_opt._inserted_poses) > 0:
        print(f"    Pose symbols: {sorted([int(gtsam.Symbol(s).index()) for s in tracking_opt._inserted_poses])}")
else:
    print("  ERROR: tracking_opt does not have _inserted_poses attribute!")
    tracking_opt._inserted_poses = set()

if hasattr(tracking_opt, '_initialized'):
    print(f"  Initialized: {tracking_opt._initialized}")
else:
    print("  ERROR: tracking_opt does not have _initialized attribute!")
    tracking_opt._initialized = False

if hasattr(tracking_opt, '_prior_noise'):
    print(f"  Has _prior_noise: ✓")
else:
    print("  ERROR: tracking_opt does not have _prior_noise attribute!")
    # Initialize from config
    if is_target_keyframe:
        prior_noise_param = pipeline_cfg.global_optimizer.params.prior_noise_param
    else:
        prior_noise_param = pipeline_cfg.local_optimizer.params.prior_noise_param
    tracking_opt._prior_noise = gtsam.noiseModel.Diagonal.Sigmas(
        np.array(prior_noise_param, dtype=float)
    )

if hasattr(tracking_opt, '_lm_params'):
    print(f"  Has _lm_params: ✓")
else:
    print("  ERROR: tracking_opt does not have _lm_params attribute!")
    # Initialize from config
    tracking_opt._lm_params = gtsam.LevenbergMarquardtParams()
    if is_target_keyframe:
        cfg_params = pipeline_cfg.global_optimizer.params
    else:
        cfg_params = pipeline_cfg.local_optimizer.params
    tracking_opt._lm_params.setMaxIterations(cfg_params.max_iterations)
    tracking_opt._lm_params.setRelativeErrorTol(cfg_params.relative_error_tol)
    tracking_opt._lm_params.setAbsoluteErrorTol(cfg_params.absolute_error_tol)
    tracking_opt._lm_params.setlambdaInitial(1e-1)
    tracking_opt._lm_params.setVerbosityLM("SUMMARY")

print(f"\n✓ Optimizer state verified. Graph and values are ready for tracking.")

# Run optimization with tracking
print(f"\nRunning optimization with per-iteration tracking for frame {FRAME_NUMBER}...")
print("(This will add the frame to the graph and track each iteration)")

# Debug: Check optimizer state before tracking
print(f"Debug: Optimizer initialized: {hasattr(tracking_opt, '_initialized') and tracking_opt._initialized}")
print(f"Debug: Graph size: {tracking_opt._graph.size() if hasattr(tracking_opt, '_graph') else 'N/A'}")
print(f"Debug: Values size: {tracking_opt._values.size() if hasattr(tracking_opt, '_values') else 'N/A'}")
if hasattr(tracking_opt, '_inserted_poses'):
    print(f"Debug: Inserted poses: {len(tracking_opt._inserted_poses)}")
    if len(tracking_opt._inserted_poses) > 0:
        print(f"Debug: Inserted pose IDs: {sorted([int(gtsam.Symbol(s).index()) for s in tracking_opt._inserted_poses])}")
print(f"Debug: Frame ID in ObjectFrameData: {target_object_frame_data.frame_id}")
if hasattr(tracking_opt, '_inserted_poses'):
    Xi_check = gtsam.symbol("x", target_object_frame_data.frame_id)
    print(f"Debug: Frame already in graph: {Xi_check in tracking_opt._inserted_poses}")
    
# Additional debug for keyframes
if is_target_keyframe:
    print(f"Debug: Number of keyframes processed before target: {kf_idx_counter}")
    print(f"Debug: Previous keyframe frame IDs: {keyframe_frame_ids}")

# Call the iteration tracker
opt_result = iteration_tracker.optimize_with_tracking(target_object_frame_data)

# Extract iteration data
iteration_errors = [d['error'] for d in iteration_tracker.iteration_data]
iteration_changes = [d.get('error_change', 0) for d in iteration_tracker.iteration_data]
iteration_poses = [d.get('values') for d in iteration_tracker.iteration_data]

if len(iteration_errors) > 0:
    # Check if this was the first frame (no optimization)
    first_frame_note = iteration_tracker.iteration_data[0].get('note', '')
    if first_frame_note:
        print(f"\n{first_frame_note}")
        print(f"Frame added to graph with prior factor. Graph size: {tracking_opt._graph.size() if hasattr(tracking_opt, '_graph') else 'N/A'}")
        print("Note: First frame in graph does not get optimized - it only establishes the reference frame.")
    else:
        print(f"\nOptimization completed. {len(iteration_errors)} iterations tracked.")
        print(f"Initial error: {iteration_errors[0]:.6f}")
        print(f"Final error: {iteration_errors[-1]:.6f}")
        if iteration_errors[0] > 0:
            print(f"Error reduction: {iteration_errors[0] - iteration_errors[-1]:.6f} ({100*(iteration_errors[0] - iteration_errors[-1])/iteration_errors[0]:.2f}%)")
        else:
            print("Initial error was zero - no reduction possible")
    
    # Update trajectories with optimized result
    if opt_result is not None and opt_result.pose_optimized is not None:
        traj_frontend.append(target_fd['pose_frontend'].copy())
        traj_local.append(opt_result.pose_optimized.copy())
        traj_global.append(opt_result.pose_optimized.copy())
        if target_fd['gt_pose'] is not None:
            traj_gt.append(target_fd['gt_pose'].copy())
        else:
            if len(traj_gt) > 0:
                traj_gt.append(traj_gt[-1].copy())
            else:
                traj_gt.append(np.eye(4))
else:
    print("\nWarning: No iteration data was collected. The frame may have already been in the graph.")
    # Still add to trajectories
    traj_frontend.append(target_fd['pose_frontend'].copy())
    traj_local.append(target_fd['pose_frontend'].copy())
    traj_global.append(target_fd['pose_frontend'].copy())
    if target_fd['gt_pose'] is not None:
        traj_gt.append(target_fd['gt_pose'].copy())
    else:
        if len(traj_gt) > 0:
            traj_gt.append(traj_gt[-1].copy())
        else:
            traj_gt.append(np.eye(4))


## Visualizations

### Per-Iteration Optimization Progress


In [ ]:
# Visualize per-iteration optimization progress
if len(iteration_errors) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    
    # Error over iterations
    ax = axes[0, 0]
    iterations = [d['iteration'] for d in iteration_tracker.iteration_data]
    ax.plot(iterations, iteration_errors, 'b-o', linewidth=2, markersize=6)
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Error')
    ax.set_title(f'Optimization Error vs Iteration (Frame {FRAME_NUMBER})')
    ax.set_yscale('log')
    ax.grid(True, alpha=0.3)

# Error change per iteration
ax = axes[0, 1]
if len(iteration_changes) > 1:
    ax.plot(iterations[1:], iteration_changes[1:], 'g-o', linewidth=2, markersize=6)
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Error Change')
    ax.set_title('Error Reduction per Iteration')
    ax.grid(True, alpha=0.3)
    ax.axhline(y=0, color='r', linestyle='--', alpha=0.5)

# Relative error change
ax = axes[1, 0]
if len(iteration_errors) > 1:
    relative_changes = []
    for i in range(1, len(iteration_errors)):
        if iteration_errors[i-1] > 0:
            rel_change = (iteration_errors[i-1] - iteration_errors[i]) / iteration_errors[i-1] * 100
            relative_changes.append(rel_change)
        else:
            relative_changes.append(0)
    ax.plot(iterations[1:], relative_changes, 'r-o', linewidth=2, markersize=6)
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Relative Error Reduction (%)')
    ax.set_title('Relative Error Reduction per Iteration')
    ax.grid(True, alpha=0.3)

# Cumulative error reduction
ax = axes[1, 1]
if len(iteration_errors) > 0:
    initial_error = iteration_errors[0]
    cumulative_reduction = [(initial_error - err) / initial_error * 100 for err in iteration_errors]
    ax.plot(iterations, cumulative_reduction, 'm-o', linewidth=2, markersize=6)
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Cumulative Error Reduction (%)')
    ax.set_title('Cumulative Error Reduction')
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
else:
    print("No iteration data available for visualization.")
    print("This may happen if:")
    print("  - The frame was already optimized in the graph")
    print("  - The optimization was skipped (e.g., first frame)")
    print("  - An error occurred during optimization tracking")


### Per-Iteration Pose Changes


In [ ]:
# Extract pose changes per iteration
if any(p is not None for p in iteration_poses):
    valid_poses = [p for p in iteration_poses if p is not None]
    valid_iterations = [i for i, p in enumerate(iteration_poses) if p is not None]
    
    if len(valid_poses) > 1:
        # Extract translation and rotation components
        translations = np.array([p[:3, 3] for p in valid_poses])
        rotations = np.array([R.from_matrix(p[:3, :3]).as_euler('xyz', degrees=True) for p in valid_poses])
        
        fig, axes = plt.subplots(2, 3, figsize=(18, 10))
        
        # Translation components
        for i, (ax, label, color) in enumerate(zip(axes[0], ['X', 'Y', 'Z'], ['r', 'g', 'b'])):
            ax.plot(valid_iterations, translations[:, i], f'{color}-o', linewidth=2, markersize=6)
            ax.set_xlabel('Iteration')
            ax.set_ylabel(f'Translation {label} (m)')
            ax.set_title(f'Translation {label} per Iteration')
            ax.grid(True, alpha=0.3)
        
        # Rotation components
        for i, (ax, label, color) in enumerate(zip(axes[1], ['Roll', 'Pitch', 'Yaw'], ['r', 'g', 'b'])):
            ax.plot(valid_iterations, rotations[:, i], f'{color}-o', linewidth=2, markersize=6)
            ax.set_xlabel('Iteration')
            ax.set_ylabel(f'Rotation {label} (deg)')
            ax.set_title(f'Rotation {label} per Iteration')
            ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        # Compute pose differences between iterations
        pose_diffs_trans = []
        pose_diffs_rot = []
        for i in range(1, len(valid_poses)):
            # Translation difference
            trans_diff = np.linalg.norm(translations[i] - translations[i-1])
            pose_diffs_trans.append(trans_diff)
            
            # Rotation difference
            R1 = valid_poses[i-1][:3, :3]
            R2 = valid_poses[i][:3, :3]
            R_diff = R1.T @ R2
            tr = np.trace(R_diff)
            theta = np.arccos(np.clip((tr - 1)/2, -1, 1))
            pose_diffs_rot.append(np.degrees(theta))
        
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        ax = axes[0]
        ax.plot(valid_iterations[1:], pose_diffs_trans, 'b-o', linewidth=2, markersize=6)
        ax.set_xlabel('Iteration')
        ax.set_ylabel('Translation Change (m)')
        ax.set_title('Translation Change per Iteration')
        ax.set_yscale('log')
        ax.grid(True, alpha=0.3)
        
        ax = axes[1]
        ax.plot(valid_iterations[1:], pose_diffs_rot, 'r-o', linewidth=2, markersize=6)
        ax.set_xlabel('Iteration')
        ax.set_ylabel('Rotation Change (deg)')
        ax.set_title('Rotation Change per Iteration')
        ax.set_yscale('log')
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
else:
    print("No pose data available for per-iteration tracking.")


### Trajectory Visualization


In [ ]:
# Convert trajectories to numpy arrays
traj_frontend = np.array(traj_frontend)
traj_local = np.array(traj_local)
traj_global = np.array(traj_global)
traj_gt = np.array(traj_gt)

# 3D Trajectory Plot
fig = plt.figure(figsize=(14, 10))
ax = fig.add_subplot(111, projection='3d')

# Plot trajectories
ax.plot(traj_frontend[:, 0, 3], traj_frontend[:, 1, 3], traj_frontend[:, 2, 3], 
        'r.-', alpha=0.6, label='Frontend', linewidth=1.5, markersize=3, markevery=5)
ax.plot(traj_local[:, 0, 3], traj_local[:, 1, 3], traj_local[:, 2, 3], 
        'b.-', alpha=0.6, label='Local Opt', linewidth=1.5, markersize=3, markevery=5)

# Highlight keyframes
if len(keyframe_frame_ids) > 0:
    kf_indices = [i for i, fid in enumerate(meta_data['frame_id'][:frame_idx+1]) if fid in keyframe_frame_ids]
    kf_array = np.array(kf_indices)
    if len(kf_array) > 0:
        ax.scatter(traj_global[kf_array, 0, 3], 
                   traj_global[kf_array, 1, 3], 
                   traj_global[kf_array, 2, 3], 
                   c='m', marker='o', s=150, alpha=0.9, label='Global Opt (Keyframes)', 
                   edgecolors='darkmagenta', linewidths=2, zorder=5)

# Highlight target frame
ax.scatter(traj_local[frame_idx, 0, 3], 
           traj_local[frame_idx, 1, 3], 
           traj_local[frame_idx, 2, 3], 
           c='orange', marker='*', s=500, alpha=1.0, label=f'Target Frame {FRAME_NUMBER}', 
           edgecolors='red', linewidths=2, zorder=10)

if len(traj_gt) > 0:
    ax.plot(traj_gt[:, 0, 3], traj_gt[:, 1, 3], traj_gt[:, 2, 3], 
            'g.-', label='GT', linewidth=2, markersize=4, markevery=5, alpha=0.7)

ax.set_xlabel('X (m)')
ax.set_ylabel('Y (m)')
ax.set_zlabel('Z (m)')
ax.legend()
ax.set_title(f"3D Trajectory up to Frame {FRAME_NUMBER}")
plt.show()

# Translation components over time
fig, axes = plt.subplots(3, 1, figsize=(15, 10))
frames = np.arange(len(traj_frontend))

for i, (ax, label) in enumerate(zip(axes, ['X', 'Y', 'Z'])):
    ax.plot(frames, traj_frontend[:, i, 3], 'r-', alpha=0.7, label='Frontend', linewidth=2)
    ax.plot(frames, traj_local[:, i, 3], 'b-', label='Local Opt', linewidth=2)
    
    # Highlight keyframes
    if len(keyframe_frame_ids) > 0:
        kf_indices = [j for j, fid in enumerate(meta_data['frame_id'][:frame_idx+1]) if fid in keyframe_frame_ids]
        if len(kf_indices) > 0:
            ax.scatter([frames[j] for j in kf_indices], 
                       [traj_global[j, i, 3] for j in kf_indices],
                       c='m', marker='o', s=100, alpha=0.9, label='Global Opt (Keyframes)',
                       edgecolors='darkmagenta', linewidths=1.5, zorder=5)
    
    # Highlight target frame
    ax.scatter(frames[frame_idx], traj_local[frame_idx, i, 3],
               c='orange', marker='*', s=300, alpha=1.0, label=f'Target Frame {FRAME_NUMBER}',
               edgecolors='red', linewidths=2, zorder=10)
    
    if len(traj_gt) > 0:
        ax.plot(frames, traj_gt[:, i, 3], 'g-', label='GT', linewidth=2, alpha=0.7)
    
    ax.set_title(f"Translation {label} over Time")
    ax.set_xlabel("Frame")
    ax.set_ylabel(f"{label} (m)")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


### Error Analysis


In [ ]:
# Compute errors if GT is available
if len(traj_gt) > 0 and not np.allclose(traj_gt[0], np.eye(4)):
    # Translation errors
    trans_err_frontend = np.linalg.norm(traj_frontend[:, :3, 3] - traj_gt[:, :3, 3], axis=1)
    trans_err_local = np.linalg.norm(traj_local[:, :3, 3] - traj_gt[:, :3, 3], axis=1)
    trans_err_global = np.linalg.norm(traj_global[:, :3, 3] - traj_gt[:, :3, 3], axis=1)
    
    # Rotation errors
    def compute_rotation_error(pose1, pose2):
        R1 = pose1[:3, :3]
        R2 = pose2[:3, :3]
        R_diff = R1.T @ R2
        tr = np.trace(R_diff)
        theta = np.arccos(np.clip((tr - 1)/2, -1, 1))
        return np.degrees(theta)
    
    rot_err_frontend = [compute_rotation_error(traj_frontend[i], traj_gt[i]) for i in range(len(traj_frontend))]
    rot_err_local = [compute_rotation_error(traj_local[i], traj_gt[i]) for i in range(len(traj_local))]
    rot_err_global = [compute_rotation_error(traj_global[i], traj_gt[i]) for i in range(len(traj_global))]
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    
    # Translation error over time
    ax = axes[0, 0]
    ax.plot(frames, trans_err_frontend, 'r-', alpha=0.7, label='Frontend', linewidth=2)
    ax.plot(frames, trans_err_local, 'b-', alpha=0.7, label='Local Opt', linewidth=2)
    ax.plot(frames, trans_err_global, 'm-', alpha=0.7, label='Global Opt', linewidth=2)
    ax.scatter(frames[frame_idx], trans_err_local[frame_idx],
               c='orange', marker='*', s=300, alpha=1.0, label=f'Target Frame {FRAME_NUMBER}',
               edgecolors='red', linewidths=2, zorder=10)
    ax.set_xlabel('Frame')
    ax.set_ylabel('Translation Error (m)')
    ax.set_title('Translation Error vs GT')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Rotation error over time
    ax = axes[0, 1]
    ax.plot(frames, rot_err_frontend, 'r-', alpha=0.7, label='Frontend', linewidth=2)
    ax.plot(frames, rot_err_local, 'b-', alpha=0.7, label='Local Opt', linewidth=2)
    ax.plot(frames, rot_err_global, 'm-', alpha=0.7, label='Global Opt', linewidth=2)
    ax.scatter(frames[frame_idx], rot_err_local[frame_idx],
               c='orange', marker='*', s=300, alpha=1.0, label=f'Target Frame {FRAME_NUMBER}',
               edgecolors='red', linewidths=2, zorder=10)
    ax.set_xlabel('Frame')
    ax.set_ylabel('Rotation Error (deg)')
    ax.set_title('Rotation Error vs GT')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Error at target frame
    ax = axes[1, 0]
    categories = ['Frontend', 'Local Opt', 'Global Opt']
    trans_errors = [trans_err_frontend[frame_idx], trans_err_local[frame_idx], trans_err_global[frame_idx]]
    colors = ['r', 'b', 'm']
    bars = ax.bar(categories, trans_errors, color=colors, alpha=0.7)
    ax.set_ylabel('Translation Error (m)')
    ax.set_title(f'Translation Error at Frame {FRAME_NUMBER}')
    ax.grid(True, alpha=0.3, axis='y')
    for bar, err in zip(bars, trans_errors):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{err:.4f}', ha='center', va='bottom')
    
    # Rotation error at target frame
    ax = axes[1, 1]
    rot_errors = [rot_err_frontend[frame_idx], rot_err_local[frame_idx], rot_err_global[frame_idx]]
    bars = ax.bar(categories, rot_errors, color=colors, alpha=0.7)
    ax.set_ylabel('Rotation Error (deg)')
    ax.set_title(f'Rotation Error at Frame {FRAME_NUMBER}')
    ax.grid(True, alpha=0.3, axis='y')
    for bar, err in zip(bars, rot_errors):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{err:.2f}', ha='center', va='bottom')
    
    plt.tight_layout()
    plt.show()
    
    # Print statistics
    print("\n=== Error Statistics ===")
    print(f"Frame {FRAME_NUMBER} Translation Errors:")
    print(f"  Frontend: {trans_err_frontend[frame_idx]:.6f} m")
    print(f"  Local Opt: {trans_err_local[frame_idx]:.6f} m")
    print(f"  Global Opt: {trans_err_global[frame_idx]:.6f} m")
    print(f"\nFrame {FRAME_NUMBER} Rotation Errors:")
    print(f"  Frontend: {rot_err_frontend[frame_idx]:.2f} deg")
    print(f"  Local Opt: {rot_err_local[frame_idx]:.2f} deg")
    print(f"  Global Opt: {rot_err_global[frame_idx]:.2f} deg")
else:
    print("GT data not available for error analysis.")


### Frame Data Summary


In [ ]:
# Print summary information about the target frame
print("=" * 60)
print(f"FRAME {FRAME_NUMBER} SUMMARY")
print("=" * 60)
print(f"\nFrame Index: {frame_idx}")
print(f"Is Keyframe: {is_target_keyframe}")
print(f"\nRegistration Data:")
print(f"  Number of 3D points: {len(target_fd['cur_3d'])}")
print(f"  Number of inliers: {np.sum(target_fd['inliers'])}")
print(f"  Mean residual: {np.mean(target_fd['residuals']):.6f}")
print(f"  Max residual: {np.max(target_fd['residuals']) if len(target_fd['residuals']) > 0 else 0:.6f}")
print(f"\nPoses:")
print(f"  Frontend pose translation: {target_fd['pose_frontend'][:3, 3]}")
print(f"  Local opt pose translation: {target_fd['pose_local'][:3, 3]}")
if opt_result is not None:
    print(f"  Optimized pose translation: {opt_result.pose_optimized[:3, 3]}")
if target_fd['gt_pose'] is not None:
    print(f"  GT pose translation: {target_fd['gt_pose'][:3, 3]}")

print(f"\nOptimization Iterations: {len(iteration_errors)}")
if len(iteration_errors) > 0:
    print(f"  Initial error: {iteration_errors[0]:.6f}")
    print(f"  Final error: {iteration_errors[-1]:.6f}")
    if iteration_errors[0] > 0:
        print(f"  Total reduction: {iteration_errors[0] - iteration_errors[-1]:.6f} ({100*(iteration_errors[0] - iteration_errors[-1])/iteration_errors[0]:.2f}%)")
    else:
        print("  Initial error was zero - no reduction possible")
else:
    print("  No iteration data collected.")
    print("  Possible reasons:")
    print("    - Frame was already in the graph")
    print("    - Optimization was skipped (e.g., first frame)")
    print("    - Error during tracking")
